In [1]:
import chromadb
client = chromadb.Client()

In [2]:
from datetime import datetime

# check if the collection already exists
existing_collections = client.list_collections()
if any(col.name == "reviews" for col in existing_collections):
    collection = client.get_collection("reviews")
    print("Collection 'reviews' already exists.")
else:
    collection = client.create_collection(
        name = "reviews",
        metadata = {
            "hnsw:space": "cosine",
            "description": "A collection of reviews for various products.",
            "created": str(datetime.now())
        }
    )



In [3]:
# Add mixed reviews to the collection

collection.add(
    documents = [
        "This product is great! I love it.",
        "Terrible quality, broke after one use.",
        "Decent product for the price.",
        "Exceeded my expectations, highly recommend!",
        "Not worth the money, very disappointed.",
        "Average quality, nothing special.",
        "Fantastic product, will buy again.",
    ],
    metadatas = [
        {"product_id": "A123", "product_category": "Electronics", "rating": 5},
        {"product_id": "B456", "product_category": "Electronics", "rating": 1},
        {"product_id": "C789", "product_category": "Electronics", "rating": 3},
        {"product_id": "D012", "product_category": "Apparel", "rating": 5},
        {"product_id": "E345", "product_category": "Electronics", "rating": 2},
        {"product_id": "F678", "product_category": "Service", "rating": 3},
        {"product_id": "G901", "product_category": "Apparel", "rating": 4}
    ],
    ids = ["1", "2", "3", "4", "5", "6", "7"]
)

In [4]:
collection.count()  # Should return 7, as we added 7 reviews

7

In [5]:
collection.peek()  # Should return the first 5 reviews in the collection

{'ids': ['1', '2', '3', '4', '5', '6', '7'],
 'embeddings': array([[-0.05339955,  0.04499586, -0.00733098, ...,  0.0847749 ,
          0.04904888,  0.07034498],
        [-0.05550962,  0.01987089, -0.0103188 , ..., -0.05538797,
          0.02481297,  0.119896  ],
        [-0.05176774,  0.05773162,  0.02595101, ..., -0.04907491,
         -0.03382324,  0.05588244],
        ...,
        [-0.03750031,  0.06193686,  0.01247057, ..., -0.17225935,
         -0.05694237,  0.10603698],
        [-0.05381998, -0.00083367,  0.03816707, ..., -0.02105257,
         -0.04957383,  0.02069788],
        [-0.04918627, -0.02223618,  0.01258727, ..., -0.01861173,
         -0.04522941,  0.07610334]], shape=(7, 384)),
 'documents': ['This product is great! I love it.',
  'Terrible quality, broke after one use.',
  'Decent product for the price.',
  'Exceeded my expectations, highly recommend!',
  'Not worth the money, very disappointed.',
  'Average quality, nothing special.',
  'Fantastic product, will buy aga

In [6]:
collection.query(
    query_texts=["I love this product!"],
    n_results=2
)

{'ids': [['1', '7']],
 'embeddings': None,
 'documents': [['This product is great! I love it.',
   'Fantastic product, will buy again.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'product_id': 'A123',
    'rating': 5,
    'product_category': 'Electronics'},
   {'product_category': 'Apparel', 'product_id': 'G901', 'rating': 4}]],
 'distances': [[0.151178777217865, 0.5182503461837769]]}

In [7]:
collection.query(
    query_texts=["This product is terrible."],
    n_results=2,
    where={"rating": 1}
)

{'ids': [['2']],
 'embeddings': None,
 'documents': [['Terrible quality, broke after one use.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'product_category': 'Electronics',
    'rating': 1,
    'product_id': 'B456'}]],
 'distances': [[0.6082271337509155]]}

In [8]:
collection.update(
    ids=["7"],
    metadatas=[{"product_id": "G901", "product_category": "Apparel", "rating": "5"}]
)

In [12]:
collection.get(
    ids=["7"]
)

{'ids': ['7'],
 'embeddings': None,
 'documents': ['Fantastic product, will buy again.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'product_id': 'G901',
   'rating': '5',
   'product_category': 'Apparel'}]}

In [13]:
collection.delete(
    ids=["7"]
)

{'deleted': 1}

In [14]:
client.delete_collection("reviews")  # Deletes the collection named "reviews"